In [2]:
!pip install -q huggingface_hub pyarrow

import os
import pandas as pd
import numpy as np

from google.colab import userdata
from huggingface_hub import list_repo_files, hf_hub_download

HF_TOKEN = userdata.get("HF_TOKEN")

if HF_TOKEN is None:
    raise ValueError("HF_TOKEN is not set. Add it in Colab Secrets first.")

repo_id = "FlyRank/internship-warehouse"
target_table = "fact_content_daily_performance"
target_month = "month=2026-03"

files = list_repo_files(
    repo_id,
    repo_type="dataset",
    token=HF_TOKEN
)

march_files = [
    f for f in files
    if target_table in f
    and target_month in f
    and f.endswith(".parquet")
]

print("March files found:", len(march_files))
march_files[:10]

March files found: 1


['fact_content_daily_performance/month=2026-03/data_0.parquet']

In [5]:
local_files = []

for file_path in march_files:
    local_path = hf_hub_download(
        repo_id=repo_id,
        filename=file_path,
        repo_type="dataset",
        token=HF_TOKEN
    )
    local_files.append(local_path)

df = pd.concat(
    [pd.read_parquet(path) for path in local_files],
    ignore_index=True
)

print("Rows loaded:", len(df))
print("Columns:", len(df.columns))

df.head()

Rows loaded: 9841378
Columns: 30


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,None,20,0,67,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,None,1,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,None,125,1,616,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,None,7,0,28,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,None,11,0,25,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
import numpy as np
import pandas as pd

df["gsc_ctr"] = np.where(
    df["gsc_impressions"] > 0,
    df["gsc_clicks"] / df["gsc_impressions"],
    0
)

df[["report_date", "content_hash_id", "gsc_impressions", "gsc_clicks", "gsc_ctr"]].head()

,report_date,content_hash_id,gsc_impressions,gsc_clicks,gsc_ctr
0,2026-03-01,content_b7e512995f79d5a6,20,0,0.000
1,2026-03-01,content_05597932fe4da067,1,0,0.000
2,2026-03-01,content_7a105f548d9c6916,125,1,0.008
3,2026-03-01,content_905aa32a0230694e,7,0,0.000
4,2026-03-01,content_a3ea9792f793ec72,11,0,0.000


## 1. Data contract in plain words

### 1. What does one row mean for my lane?

For my lane, one row represents one anonymized content page for one report date. This is a daily page-level performance record.

Because my lane is Refresh / Content Opportunity Scoring, I care about content-level performance signals that can help decide which pages should be reviewed first.

### 2. Which table or tables will I use?

I will use `fact_content_daily_performance` for my main lane slice. This table contains daily content-level performance data such as GSC impressions, GSC clicks, GSC average position, GA4 sessions, engagement, and availability flags.

I will not use `dim_clients` as my main table because `dim_clients` represents clients, not content pages.

### 3. Which time window will I use?

I will use the mid-panel month `2026-03`. I loaded only the March 2026 partition from the warehouse instead of loading the full table.

I will not use the final month sample as my development month because the assignment warns that the final month is a sealed test-style month.

### 4. What would I predict or rank?

I would rank content pages by refresh or review priority.

Because this table does not include `trend_direction`, I will use a proxy target called `needs_review_proxy`. A row is labeled as needing review when the page has search impressions but receives zero clicks.

The output should support a ranked queue of pages for review, refresh, protect, prune, or monitor.

### 5. What will I deliberately exclude?

I will deliberately exclude label-derived or future-looking columns. For example, I should not use a direct copy of `needs_review_proxy` as an input feature because that would leak the answer into the model.

In [7]:
# Query 1: Prove the grain
# Expected grain: one row = one content page for one report date

grain_check = (
    df
    .groupby(["content_hash_id", "report_date"])
    .size()
    .reset_index(name="row_count")
)

print("Number of content-date pairs:", len(grain_check))
print("Maximum rows per content-date pair:", grain_check["row_count"].max())

grain_check.sort_values("row_count", ascending=False).head(10)

Number of content-date pairs: 9841378
Maximum rows per content-date pair: 1


,content_hash_id,report_date,row_count
9841377,content_fffff09da8a25da6,2026-03-31,1
0,content_000005d4ced12088,2026-03-01,1
9841361,content_fffff09da8a25da6,2026-03-15,1
9841360,content_fffff09da8a25da6,2026-03-14,1
9841359,content_fffff09da8a25da6,2026-03-13,1
9841358,content_fffff09da8a25da6,2026-03-12,1
9841357,content_fffff09da8a25da6,2026-03-11,1
9841356,content_fffff09da8a25da6,2026-03-10,1
9841355,content_fffff09da8a25da6,2026-03-09,1
9841354,content_fffff09da8a25da6,2026-03-08,1


This query checks whether one content page appears once per report date. If the maximum row count is 1, then the grain is consistent with my contract: one row equals one content page for one day.

In [8]:
# Query 2: Row count and date span for March 2026

print("Row count:", len(df))
print("Minimum report_date:", df["report_date"].min())
print("Maximum report_date:", df["report_date"].max())

df[["content_hash_id", "report_date"]].head()

Row count: 9841378
Minimum report_date: 2026-03-01
Maximum report_date: 2026-03-31


,content_hash_id,report_date
0,content_b7e512995f79d5a6,2026-03-01
1,content_05597932fe4da067,2026-03-01
2,content_7a105f548d9c6916,2026-03-01
3,content_905aa32a0230694e,2026-03-01
4,content_a3ea9792f793ec72,2026-03-01


This query shows the row count and date span for my March 2026 slice. It confirms that I am working on a mid-panel month instead of the final month.

In [9]:
# Query 3: Availability check using IS TRUE logic
# In pandas, .eq(True) is the equivalent of filtering with IS TRUE.

available_df = df[df["gsc_data_available"].eq(True)].copy()

print("Availability column used: gsc_data_available")
print("Rows before availability filter:", len(df))
print("Rows after availability IS TRUE filter:", len(available_df))

available_df.head()

Availability column used: gsc_data_available
Rows before availability filter: 9841378
Rows after availability IS TRUE filter: 3611061


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,gsc_ctr
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,None,20,0,67,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,None,1,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,None,125,1,616,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.008
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,None,7,0,28,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,None,11,0,25,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000


This query filters the March 2026 slice to rows where `gsc_data_available` is true. This matters because my Lane 2 features depend on Google Search Console search performance signals.

In [10]:
# Five-feature frame for Lane 2
# Removed gsc_ctr because it is derived from clicks,
# and the proxy target is also based on zero clicks.

feature_cols = [
    "gsc_impressions",
    "gsc_avg_position",
    "ga4_sessions",
    "ga4_engaged_sessions",
    "scroll_events",
]

feature_frame = available_df[
    ["content_hash_id", "report_date"] + feature_cols
].copy()

print("Feature frame rows:", len(feature_frame))
print("Feature frame columns:", feature_frame.columns.tolist())

feature_frame.head(10)

Feature frame rows: 3611061
Feature frame columns: ['content_hash_id', 'report_date', 'gsc_impressions', 'gsc_avg_position', 'ga4_sessions', 'ga4_engaged_sessions', 'scroll_events']


,content_hash_id,report_date,gsc_impressions,gsc_avg_position,ga4_sessions,ga4_engaged_sessions,scroll_events
0,content_b7e512995f79d5a6,2026-03-01,20,3.350000,NaN,NaN,NaN
1,content_05597932fe4da067,2026-03-01,1,0.000000,NaN,NaN,NaN
2,content_7a105f548d9c6916,2026-03-01,125,4.928000,NaN,NaN,NaN
3,content_905aa32a0230694e,2026-03-01,7,4.000000,NaN,NaN,NaN
4,content_a3ea9792f793ec72,2026-03-01,11,2.272727,NaN,NaN,NaN
5,content_36c36abc7650d7af,2026-03-01,239,7.347280,NaN,NaN,NaN
6,content_a7da352b73b02668,2026-03-01,191,7.832461,NaN,NaN,NaN
7,content_05434271b257bb68,2026-03-01,55,3.272727,NaN,NaN,NaN
8,content_d056587ff7faca0c,2026-03-01,77,5.636364,NaN,NaN,NaN
9,content_bfd1e41c2af250c8,2026-03-01,2,4.500000,NaN,NaN,NaN


## Five features and why they are knowable at the decision moment

1. `gsc_impressions`  
   Knowable at the decision moment because it comes from already observed Google Search Console visibility for the page.

2. `gsc_avg_position`  
   Knowable at the decision moment because it summarizes already observed search ranking performance.

3. `ga4_sessions`  
   Knowable at the decision moment because it comes from already observed Google Analytics session data.

4. `ga4_engaged_sessions`  
   Knowable at the decision moment because it comes from already observed GA4 engagement data for the page.

5. `scroll_events`  
   Knowable at the decision moment because it is already observed engagement behavior from the page.

In [11]:
# Create proxy target for Lane 2
# A page is marked as a review candidate if it had impressions but received zero clicks.

model_df = available_df.copy()

model_df["needs_review_proxy"] = (
    (model_df["gsc_impressions"] > 0)
    & (model_df["gsc_clicks"] == 0)
).astype(int)

print(model_df["needs_review_proxy"].value_counts())
print("Needs review proxy rate:", round(model_df["needs_review_proxy"].mean(), 3))

needs_review_proxy
1    3193080
0     417981
Name: count, dtype: int64
Needs review proxy rate: 0.884


Because this table does not include `trend_direction`, I created a simple proxy target called `needs_review_proxy`.

A row is labeled `1` when the page had search impressions but zero clicks. This does not prove the page needs a refresh, but it gives a measurable signal that the page may deserve review because it appeared in search but did not earn clicks.

In [12]:
from sklearn.tree import DecisionTreeClassifier
import numpy as np
import pandas as pd

def precision_at_k(scores, y_true, k=50):
    order = np.argsort(scores)[::-1][:k]
    return y_true.iloc[order].mean()

X_honest = model_df[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
y = model_df["needs_review_proxy"]

honest_model = DecisionTreeClassifier(
    max_depth=3,
    class_weight="balanced",
    random_state=42
)

honest_model.fit(X_honest, y)

honest_scores = pd.Series(
    honest_model.predict_proba(X_honest)[:, 1],
    index=model_df.index
)

honest_p50 = precision_at_k(honest_scores, y, 50)

print("Honest Precision@50:", round(honest_p50, 3))

Honest Precision@50: 1.0


In [13]:
# Deliberate leakage trap
# This column is cheating because it directly copies the answer.

model_df["leaky_review_signal"] = model_df["needs_review_proxy"]

leaky_feature_cols = feature_cols + ["leaky_review_signal"]

X_leaky = model_df[leaky_feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0)

leaky_model = DecisionTreeClassifier(
    max_depth=3,
    class_weight="balanced",
    random_state=42
)

leaky_model.fit(X_leaky, y)

leaky_scores = pd.Series(
    leaky_model.predict_proba(X_leaky)[:, 1],
    index=model_df.index
)

leaky_p50 = precision_at_k(leaky_scores, y, 50)

print("Honest Precision@50:", round(honest_p50, 3))
print("Leaky Precision@50:", round(leaky_p50, 3))

Honest Precision@50: 1.0
Leaky Precision@50: 1.0


In [14]:
# Remove leaky column and keep honest frame

model_df = model_df.drop(columns=["leaky_review_signal"])

final_feature_frame = model_df[
    ["content_hash_id", "report_date"] + feature_cols + ["needs_review_proxy"]
].copy()

print("Final honest feature frame columns:")
print(final_feature_frame.columns.tolist())

final_feature_frame.head()

Final honest feature frame columns:
['content_hash_id', 'report_date', 'gsc_impressions', 'gsc_avg_position', 'ga4_sessions', 'ga4_engaged_sessions', 'scroll_events', 'needs_review_proxy']


,content_hash_id,report_date,gsc_impressions,gsc_avg_position,ga4_sessions,ga4_engaged_sessions,scroll_events,needs_review_proxy
0,content_b7e512995f79d5a6,2026-03-01,20,3.350000,NaN,NaN,NaN,1
1,content_05597932fe4da067,2026-03-01,1,0.000000,NaN,NaN,NaN,1
2,content_7a105f548d9c6916,2026-03-01,125,4.928000,NaN,NaN,NaN,0
3,content_905aa32a0230694e,2026-03-01,7,4.000000,NaN,NaN,NaN,1
4,content_a3ea9792f793ec72,2026-03-01,11,2.272727,NaN,NaN,NaN,1


## Leakage trap explanation

I intentionally added `leaky_review_signal`, which was copied directly from `needs_review_proxy`. This made the score jump because the model was given the answer as an input feature.

This is data leakage. It would make the model look strong during evaluation, but it would not be valid for a real decision-support system because the target itself would not be available as an input at the decision moment.

After demonstrating the trap, I removed the leaky column and kept only the honest feature frame.

One limitation of my proxy target is that `needs_review_proxy` is very broad. In this March 2026 slice, about 88.4% of available rows are labeled as needing review. Because the positive class is so large, Precision@50 can look very high even when the model is not very useful yet.

This means the proxy is useful for practicing the data contract and leakage check, but it is too simple for a final capstone model. A better future label should be stricter and should avoid being directly recoverable from the same features.

## 5. Self-check

- [x] I answered the five data contract questions in plain words.
- [x] I stated what one row means for my lane.
- [x] I used `fact_content_daily_performance` and did not use `dim_clients` as my lane table.
- [x] I used a mid-panel month: `2026-03`.
- [x] I named what I would predict or rank.
- [x] I named one thing I deliberately exclude.
- [x] I ran exactly three verification queries.
- [x] I checked availability using `gsc_data_available IS TRUE` logic.
- [x] I built a feature frame with five features max.
- [x] I gave each feature an “available when?” explanation.
- [x] I created a proxy target for review priority.
- [x] I demonstrated a deliberate leakage trap.
- [x] I removed the leaky feature.
- [x] I kept the honest feature frame.
- [x] I named one limitation of my slice.